# LLM Redis Worker (Jupyter DataLab)

Воркер LLM-моста redis-bridge: слушает stream заявок в Redis, исполняет их
против целевого LLM-бэкенда (GigaChat / OpenAI-совместимый) и возвращает
ответ. Протокол — `docs/integrations/redis-llm-bridge.md`.

**Запуск:** Run All. Ячейка запуска откажется стартовать прослушку (упадёт с
понятной ошибкой), если не настроена ни одна цель LLM или недоступен Redis —
см. подсказки в выводе ячейки конфигурации. Остановка: прервать ячейку
запуска (Kernel → Interrupt) — heartbeat-ключ удалится, приложение мгновенно
увидит «воркер недоступен».

## Где какие настройки живут

Частая путаница: «в `.env` приложения указаны `CHAT__API_BASE` и
`CHAT__API_KEY`, но на SDP таких переменных нет». Их там и не должно быть.

| Что | Где задаётся | Кому нужно |
|---|---|---|
| Адрес и токен GigaChat (`GIGACHAT_API_URL`, `JPY_API_TOKEN`) | окружение DataLab или ячейка «Переменные окружения» ниже | **только воркеру** (этому ноутбуку) |
| Адрес и ключ OpenAI-совместимого сервера (`OPENAI_API_URL`, `OPENAI_API_KEY`) | ячейка «Переменные окружения» ниже — **на DataLab этих переменных нет из коробки** (в отличие от `JPY_API_TOKEN`), заполнять вручную | **только воркеру** |
| Дефолтная модель на случай, если SDP её не передаст (`GIGACHAT_MODEL`, `OPENAI_MODEL`) | там же, опционально | **только воркеру** |
| Адрес Redis моста (`BRIDGE_REDIS_*`) | там же | **только воркеру** |
| Маршрут (`CHAT__PROFILE`) | `.env` приложения на SDP | **только приложению** |
| Модель (`CHAT__MODEL`) | `.env` приложения, опционально | для `redis-bridge,*` опциональна — пусто → берётся `GIGACHAT_MODEL`/`OPENAI_MODEL` воркера; для ПРЯМЫХ HTTP-маршрутов обязательна |
| `CHAT__API_BASE` / `CHAT__API_KEY` | `.env` приложения | только для ПРЯМЫХ HTTP-маршрутов (`CHAT__PROFILE=gigachat` / `openai`). Для `redis-bridge,*` не используются — можно оставить пустыми |

Приложение на SDP не ходит в LLM напрямую и токена не знает: оно кладёт
заявку в Redis, а в бэкенд стучится воркер — отсюда и разделение.

**Токен DataLab меняется при каждом запуске контейнера**, поэтому его нельзя
зафиксировать в `.env` приложения. Здесь это не проблема: воркер читает его
из окружения в момент запуска, то есть всегда свежий.

Приоритет значений в ячейке «Переменные окружения»: **сначала настоящее
окружение, потом то, что вписано в ячейку**. Вписывать в ячейку нужно только
то, чего в окружении нет. Что откуда приехало — видно в таблице, которую
печатает ячейка конфигурации.


## Быстрый тест на DEV (локально)

Локальный воркер + локальный Redis + облачный OpenAI-совместимый бэкенд (например, cloud.ru), без DataLab.

1. В `.env` — `CHAT__PROFILE=redis-bridge,openai`, перезапустить приложение.
   - Опционально: fallback напрямую в OpenAI-совместимый сервер (минуя воркер) — `CHAT__FALLBACK_PROFILE=openai` + `CHAT__FALLBACK_API_BASE`/`CHAT__FALLBACK_API_KEY` в `.env`.
2. Ниже, в ячейке «Переменные окружения» — заполнить в `OVERRIDES` поля `OPENAI_API_URL`/`OPENAI_API_KEY` (значение ключа — `CHAT__API_KEY` из `.env`) и `BRIDGE_REDIS_HOST=127.0.0.1`, `BRIDGE_REDIS_PORT=6379` (дефолты в ноутбуке — ПРОМ-овские).
   - Альтернатива: не трогать эту ячейку, а задать `$env:OPENAI_API_URL` и т.д. в PowerShell **до** запуска Jupyter из того же окна — дочерний процесс наследует переменные только от того шелла, что его запустил. Если ноутбук уже открыт в другом kernel (например, через VS Code) — так они не долетят, работает только правка `OVERRIDES` в ячейке ниже. Что откуда приехало, видно в таблице «Переменные воркера», которую печатает ячейка конфигурации.
3. Run All. В ячейке конфигурации должно появиться `доступные цели: ['openai']`.
4. Проверка в Redis (отдельное окно PowerShell):
   - `redis-cli GET llm:bridge:worker:alive` / `redis-cli TTL llm:bridge:worker:alive` — heartbeat, TTL 45→30.
   - `redis-cli XREAD BLOCK 0 STREAMS llm:bridge:requests $` — живой просмотр заявок (открыть ДО отправки сообщения в чате).
   - `redis-cli XREVRANGE llm:bridge:requests + - COUNT 3` — последние заявки, если live-окно пропустил.
   - `redis-cli KEYS "llm:bridge:resp:*"` → `redis-cli XRANGE llm:bridge:resp:<id> - +` — ответ воркера (TTL 300 сек, смотреть сразу после ответа).
5. В чате — тумблер «База знаний ОАРБ»: **Выключен** (идёт локальный LLM-путь), отправить сообщение.
6. Бонус — мёртвый воркер: Kernel → Interrupt на ячейке запуска → отправить сообщение в чат → должна прийти мгновенная штатная ошибка (без ожидания таймаута), `GET llm:bridge:worker:alive` вернёт `(nil)`.

Весь трафик Redis целиком (вкл. поллинг приложения раз в 0.3 сек) — `redis-cli MONITOR` (шумно, для точечной диагностики).

In [ ]:
# Зависимости (раскомментировать при первом запуске, если пакетов нет)
# %pip install redis httpx


In [ ]:
"""Переменные окружения воркера.

Приоритет: СНАЧАЛА настоящее окружение (env контейнера DataLab), ПОТОМ
значения из этой ячейки. Заполнять здесь нужно только то, чего в окружении
нет — то, что уже инжектировано контейнером, ячейка не перетирает.

Токен DataLab (JPY_API_TOKEN) меняется при каждом запуске контейнера,
поэтому обычно он уже есть в окружении и трогать его тут не нужно; таблица
в следующей ячейке покажет, нашёлся он или нет.
"""
import os

# Пустая строка = «не задано, брать из окружения».
OVERRIDES = {
    # --- цель gigachat ---
    "GIGACHAT_API_URL": "",       # напр. https://gigachat.<домен>/api/v1
    "JPY_API_TOKEN": "",          # токен DataLab; обычно уже в окружении
    "GIGACHAT_MODEL": "",         # дефолт модели, если SDP не передал model в заявке
    # --- цель openai: sglang / vLLM / openrouter / cloud.ru ---
    # На DataLab OPENAI_API_URL/OPENAI_API_KEY НЕТ из коробки (в отличие от
    # JPY_API_TOKEN) — если нужна цель 'openai', заполнять здесь обязательно.
    "OPENAI_API_URL": "",         # напр. https://foundation-models.api.cloud.ru/v1
    "OPENAI_API_KEY": "",         # можно пусто: локальный sglang/vLLM без авторизации
    "OPENAI_MODEL": "",           # дефолт модели, если SDP не передал model в заявке
    # --- Redis моста (пусто → ПРОМ-дефолты 10.110.10.38:7474 без пароля) ---
    "BRIDGE_REDIS_HOST": "",
    "BRIDGE_REDIS_PORT": "",
    "BRIDGE_REDIS_PASSWORD": "",
}

# Что реально пришлось взять из ячейки — для таблицы источников ниже.
_FROM_CELL = set()
for _name, _value in OVERRIDES.items():
    if _value and not os.environ.get(_name):
        os.environ[_name] = _value
        _FROM_CELL.add(_name)


def env_source(name: str) -> str:
    """Откуда приехало значение: окружение | ячейка | не задано."""
    if not os.environ.get(name):
        return "не задано"
    return "ячейка" if name in _FROM_CELL else "окружение"


In [ ]:
"""Конфигурация. Всё берётся из env (см. ячейку выше); ниже — константы."""
import os
import socket

# Цели: имя цели = проводной формат (openai | gigachat).
# Цель публикуется в heartbeat, если задан url; token опционален
# (локальный sglang/vLLM без авторизации — легальная цель).
# model — дефолт на случай, если SDP не передал model в заявке (см. process_entry).
TARGETS = {
    "gigachat": {
        "url": os.environ.get("GIGACHAT_API_URL", ""),
        "token": os.environ.get("JPY_API_TOKEN", ""),
        "model": os.environ.get("GIGACHAT_MODEL", ""),
        "rate_limit_sec": 5.0,   # GigaChat: не чаще 1 запроса в 5 сек
    },
    "openai": {
        "url": os.environ.get("OPENAI_API_URL", ""),
        "token": os.environ.get("OPENAI_API_KEY", ""),
        "model": os.environ.get("OPENAI_MODEL", ""),
        "rate_limit_sec": 0.0,   # sglang/vLLM: без лимита
    },
}

REDIS_HOST = os.environ.get("BRIDGE_REDIS_HOST") or "10.110.10.38"
REDIS_PORT = int(os.environ.get("BRIDGE_REDIS_PORT") or "7474")
REDIS_PASSWORD = os.environ.get("BRIDGE_REDIS_PASSWORD", "") or None

KEY_PREFIX = "llm:bridge:"
CONSUMER_GROUP = "llm-workers"
WORKER_ID = f"{socket.gethostname()}:{os.getpid()}"

HEARTBEAT_INTERVAL_SEC = 15
HEARTBEAT_TTL_SEC = 45
HEALTH_CHECK_TIMEOUT_SEC = 4    # GET /models цели в heartbeat-такте
MAX_ATTEMPTS = 3            # попыток вызова LLM на заявку
RETRY_PAUSES_SEC = [5, 10, 20]
RESP_TTL_SEC = 300
HTTP_TIMEOUT_SEC = 120
# Пауза перед повтором цикла после сетевого сбоя (обрыв соединения с Redis).
RECONNECT_PAUSE_SEC = 2
# Redis сам проверяет живость соединения раз в N сек и переподключается:
# без этого простаивающее соединение может быть тихо закрыто сетью, и
# следующая команда падает по таймауту чтения (см. ячейку запуска).
REDIS_HEALTH_CHECK_INTERVAL_SEC = 30
# XAUTOCLAIM: подобрать зависшее. Worst-case живой обработки одной заявки
# ~6 мин (HTTP 120с × 3 попытки-обрыва + паузы 5/10/20с + rate-limit) —
# порог 10 мин гарантирует, что клеймится только заведомо брошенное,
# а не заявка, которую другой воркер ещё обрабатывает (иначе дубль-вызовы).
CLAIM_MIN_IDLE_MS = 600_000


def available_targets():
    return [n for n, t in TARGETS.items() if t["url"]]


def _mask(value: str) -> str:
    """Секрет в выводе: только длина и хвост."""
    return f"…{value[-4:]} (длина {len(value)})" if value else "—"


_VARS = (
    ("GIGACHAT_API_URL", False), ("JPY_API_TOKEN", True), ("GIGACHAT_MODEL", False),
    ("OPENAI_API_URL", False), ("OPENAI_API_KEY", True), ("OPENAI_MODEL", False),
    ("BRIDGE_REDIS_HOST", False), ("BRIDGE_REDIS_PORT", False),
    ("BRIDGE_REDIS_PASSWORD", True),
)
print("Переменные воркера (источник → значение):")
for _name, _is_secret in _VARS:
    _raw = os.environ.get(_name, "")
    _shown = _mask(_raw) if _is_secret else (_raw or "—")
    print(f"  {_name:<22} {env_source(_name):<10} {_shown}")

print(f"\nRedis моста: {REDIS_HOST}:{REDIS_PORT}, префикс ключей {KEY_PREFIX!r}")
print(f"Воркер {WORKER_ID}; доступные цели: {available_targets()}")
if not available_targets():
    print(
        "  ВНИМАНИЕ: ни одна цель не настроена — ячейка запуска откажется\n"
        "  стартовать прослушку. Задайте GIGACHAT_API_URL и/или OPENAI_API_URL\n"
        "  в OVERRIDES выше."
    )
if not os.environ.get("OPENAI_API_URL"):
    print(
        "  ПОДСКАЗКА: OPENAI_API_URL не задан. На DataLab такой переменной нет\n"
        "  из коробки (в отличие от JPY_API_TOKEN) — если нужна цель 'openai',\n"
        "  впишите адрес OpenAI-совместимого сервера в OVERRIDES выше."
    )
if not os.environ.get("OPENAI_API_KEY"):
    print(
        "  ПОДСКАЗКА: OPENAI_API_KEY не задан. Для локального sglang/vLLM без\n"
        "  авторизации это нормально; для cloud.ru и других серверов с токеном —\n"
        "  впишите ключ в OVERRIDES выше."
    )
for _name in available_targets():
    if not TARGETS[_name].get("model"):
        print(
            f"  ПОДСКАЗКА: {_name.upper()}_MODEL не задан. Если SDP не передаст\n"
            "  model в заявке — бэкенд, скорее всего, отклонит запрос. Впишите\n"
            "  модель по умолчанию в OVERRIDES выше, если это нужно."
        )


In [ ]:
"""Логика воркера: heartbeat + consumer.

Оба цикла переживают сетевые сбои: обрыв соединения с Redis — штатное
событие (простаивающее TCP-соединение может закрыть сеть или сам сервер),
и воркер из-за него останавливаться не должен. Раньше любая такая ошибка
поднималась из цикла в asyncio.gather и гасила воркер целиком — снаружи
это выглядело как «поработал пару минут и упал».
"""
import asyncio
import json
import time
import traceback

import httpx
import redis.asyncio as aioredis

stats = {
    "processed": 0,
    "errors": 0,
    "started_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
_last_call_at: dict[str, float] = {}


def make_redis() -> aioredis.Redis:
    # socket_timeout НЕ задаём: XREADGROUP BLOCK держит сокет дольше 5 сек.
    # health_check_interval + keepalive: соединение пингуется и переподключается
    # само, иначе после простоя первая же команда падает по таймауту чтения.
    return aioredis.Redis(
        host=REDIS_HOST, port=REDIS_PORT, password=REDIS_PASSWORD,
        decode_responses=True,
        socket_keepalive=True,
        health_check_interval=REDIS_HEALTH_CHECK_INTERVAL_SEC,
    )


def auth_headers(cfg: dict) -> dict:
    """Authorization только при заданном token (sglang/vLLM живут без него)."""
    return {"Authorization": f"Bearer {cfg['token']}"} if cfg["token"] else {}


async def check_target_health(http: httpx.AsyncClient, name: str) -> bool:
    """GET /models цели: жив ли LLM-бэкенд за воркером.

    Здоровье = ответ со статусом < 500 (401/404 значит «сервер отвечает»).
    Rate limit НЕ трогаем: это дешёвый GET, не completion.

    Ловим Exception, а не только httpx.HTTPError: кривой URL (httpx.InvalidURL),
    ошибка DNS-резолвера или SSL дают исключения вне иерархии HTTPError, и
    раньше они убивали heartbeat вместе со всем воркером.
    """
    cfg = TARGETS[name]
    try:
        resp = await http.get(
            cfg["url"].rstrip("/") + "/models",
            headers=auth_headers(cfg),
            timeout=HEALTH_CHECK_TIMEOUT_SEC,
        )
        return resp.status_code < 500
    except Exception:
        return False


async def heartbeat_loop(r: aioredis.Redis) -> None:
    """Продлевает ключ-статус каждые 15 сек — параллельно любой обработке.

    Вместе со статусом публикует target_health: health probe приложения
    закрывает circuit breaker, только когда жив не только воркер,
    но и LLM-бэкенд за ним (иначе breaker «хлопал» бы).

    Пробы целей идут параллельно (gather): последовательно каждая мёртвая
    цель добавляла бы к такту свой HEALTH_CHECK_TIMEOUT_SEC, и фаза проб
    росла бы суммой таймаутов вместо максимума по целям.
    """
    async with httpx.AsyncClient() as http:
        while True:
            try:
                targets = available_targets()
                health = await asyncio.gather(
                    *(check_target_health(http, name) for name in targets),
                )
                payload = {
                    "worker_id": WORKER_ID,
                    "started_at": stats["started_at"],
                    "last_beat": time.strftime("%Y-%m-%dT%H:%M:%S"),
                    "processed": stats["processed"],
                    "errors": stats["errors"],
                    "targets": targets,
                    "target_health": dict(zip(targets, health)),
                }
                await r.set(
                    KEY_PREFIX + "worker:alive",
                    json.dumps(payload, ensure_ascii=False),
                    ex=HEARTBEAT_TTL_SEC,
                )
            except asyncio.CancelledError:
                raise
            except Exception as exc:
                # Ключ живёт ещё HEARTBEAT_TTL_SEC — одна пропущенная публикация
                # не роняет мост, следующий такт её наверстает.
                print(f"[heartbeat] сбой: {exc!r}; повтор через "
                      f"{HEARTBEAT_INTERVAL_SEC}с")
            await asyncio.sleep(HEARTBEAT_INTERVAL_SEC)


async def call_llm(http: httpx.AsyncClient, target: str, path: str,
                   body: dict, deadline_ts: float):
    """POST к цели с rate limit и ретраями. Возвращает ("final", resp) |
    ("error", status_code, message) | ("expired", None, None)."""
    cfg = TARGETS[target]
    last_error = (502, "нет попыток")
    for attempt in range(MAX_ATTEMPTS):
        wait = cfg["rate_limit_sec"] - (time.time() - _last_call_at.get(target, 0.0))
        if wait > 0:
            await asyncio.sleep(wait)
        if time.time() > deadline_ts:
            return ("expired", None, None)
        _last_call_at[target] = time.time()
        try:
            resp = await http.post(
                cfg["url"].rstrip("/") + path,
                json=body,
                headers=auth_headers(cfg),
            )
        except httpx.HTTPError as exc:
            last_error = (502, f"сетевая ошибка: {exc!r}")
        else:
            if resp.status_code < 400:
                return ("final", resp, None)
            if resp.status_code == 429 or resp.status_code >= 500:
                last_error = (resp.status_code, resp.text[:500])
            else:  # 4xx — не ретраим
                return ("error", resp.status_code, resp.text[:500])
        if attempt < MAX_ATTEMPTS - 1:
            pause = RETRY_PAUSES_SEC[min(attempt, len(RETRY_PAUSES_SEC) - 1)]
            if time.time() + pause > deadline_ts:
                break
            print(f"  повтор через {pause}с (попытка {attempt + 2})")
            await asyncio.sleep(pause)
    return ("error", last_error[0], last_error[1])


async def process_entry(r, http, entry_id: str, fields: dict) -> None:
    req_id = fields.get("id", "?")
    target = fields.get("target", "")
    resp_key = KEY_PREFIX + "resp:" + req_id
    received_ts = time.time()

    async def reply(payload: dict) -> None:
        await r.xadd(resp_key, {"v": "1", "seq": "0",
                                "received_ts": str(received_ts), **payload})
        await r.expire(resp_key, RESP_TTL_SEC)

    try:
        deadline_ts = float(fields.get("deadline_ts", "0") or 0)
        if deadline_ts and time.time() > deadline_ts:
            print(f"[{req_id}] просрочена, пропуск")
            return
        if target not in available_targets():
            stats["errors"] += 1
            await reply({"kind": "error", "status_code": "503",
                         "message": f"цель {target!r} не настроена",
                         "started_ts": str(time.time()),
                         "finished_ts": str(time.time())})
            return
        body = json.loads(fields["body"])
        # SDP присылает model опционально: если передал — используем как есть;
        # если нет, подставляем дефолт воркера (GIGACHAT_MODEL/OPENAI_MODEL).
        if not body.get("model"):
            default_model = TARGETS[target].get("model")
            if default_model:
                body["model"] = default_model
            else:
                print(f"[{req_id}] SDP не передал model, а "
                      f"{target.upper()}_MODEL не настроен на воркере — "
                      "бэкенд, скорее всего, отклонит запрос")
        started_ts = time.time()
        outcome, a, b = await call_llm(
            http, target, fields.get("path", "/chat/completions"),
            body, deadline_ts or (time.time() + 600),
        )
        finished_ts = time.time()
        if outcome == "expired":
            print(f"[{req_id}] дедлайн истёк во время обработки")
            return
        if outcome == "final":
            stats["processed"] += 1
            await reply({"kind": "final", "status_code": str(a.status_code),
                         "body": a.text,
                         "started_ts": str(started_ts),
                         "finished_ts": str(finished_ts)})
            print(f"[{req_id}] ok за {finished_ts - started_ts:.1f}с")
        else:
            stats["errors"] += 1
            await reply({"kind": "error", "status_code": str(a),
                         "message": str(b),
                         "started_ts": str(started_ts),
                         "finished_ts": str(finished_ts)})
            print(f"[{req_id}] ошибка {a}")
    except Exception as exc:  # заявка не должна ронять воркер
        stats["errors"] += 1
        try:
            await reply({"kind": "error", "status_code": "500",
                         "message": f"внутренняя ошибка воркера: {exc!r}",
                         "started_ts": str(time.time()),
                         "finished_ts": str(time.time())})
        except Exception:
            print(f"[{req_id}] не удалось записать ответ: {exc!r}")
    finally:
        await r.xack(KEY_PREFIX + "requests", CONSUMER_GROUP, entry_id)


async def consumer_loop(r: aioredis.Redis) -> None:
    stream = KEY_PREFIX + "requests"
    try:
        await r.xgroup_create(stream, CONSUMER_GROUP, id="$", mkstream=True)
    except aioredis.ResponseError as exc:
        if "BUSYGROUP" not in str(exc):
            raise
    async with httpx.AsyncClient(timeout=HTTP_TIMEOUT_SEC) as http:
        while True:
            try:
                # 1) подобрать зависшее (упавший воркер взял и не доделал)
                _next, claimed, _deleted = await r.xautoclaim(
                    stream, CONSUMER_GROUP, WORKER_ID,
                    min_idle_time=CLAIM_MIN_IDLE_MS, start_id="0-0", count=10,
                )
                for entry_id, fields in claimed:
                    print(f"подобрана зависшая заявка {entry_id}")
                    await process_entry(r, http, entry_id, fields)
                # 2) новые заявки
                batch = await r.xreadgroup(
                    CONSUMER_GROUP, WORKER_ID, {stream: ">"}, count=1, block=5000,
                )
                for _stream_name, entries in batch or []:
                    for entry_id, fields in entries:
                        await process_entry(r, http, entry_id, fields)
            except asyncio.CancelledError:
                raise
            except (aioredis.ConnectionError, aioredis.TimeoutError) as exc:
                # Самый частый сценарий: соединение с Redis закрыли, пока
                # XREADGROUP ждал заявку, — ответ не пришёл и redis-py отдал
                # «Timeout reading from host:port». Клиент переподключится
                # на следующей команде, группа и PEL в Redis никуда не делись.
                print(f"[consumer] соединение с Redis: {exc!r}; "
                      f"повтор через {RECONNECT_PAUSE_SEC}с")
                await asyncio.sleep(RECONNECT_PAUSE_SEC)
            except Exception:
                # Любая иная ошибка цикла: печатаем и продолжаем — уронить
                # воркер целиком хуже, чем пропустить один такт.
                print("[consumer] непредвиденная ошибка цикла:")
                traceback.print_exc()
                await asyncio.sleep(RECONNECT_PAUSE_SEC)


In [ ]:
"""Запуск. Остановка — Kernel → Interrupt: heartbeat-ключ удаляется."""
import traceback

# Жёсткая проверка перед стартом прослушки: без хотя бы одной цели LLM и
# живого Redis воркер бесполезен — лучше отказаться стартовать сразу,
# чем поднять heartbeat и потом ронять каждую заявку с «цель недоступна».
if not available_targets():
    raise RuntimeError(
        "Ни одна цель LLM не настроена (нет ни GIGACHAT_API_URL, ни "
        "OPENAI_API_URL). Задайте хотя бы одну в OVERRIDES ячейки "
        "«Переменные окружения» и перезапустите Run All."
    )

_startup_redis = make_redis()
try:
    await _startup_redis.ping()
except Exception as exc:
    raise RuntimeError(
        f"Redis моста недоступен ({REDIS_HOST}:{REDIS_PORT}): {exc!r}. "
        "Проверьте BRIDGE_REDIS_HOST/PORT/PASSWORD в OVERRIDES и "
        "перезапустите Run All."
    ) from exc
finally:
    try:
        await _startup_redis.aclose()
    except Exception:
        pass

# Два отдельных клиента, а не один общий: cancel во время XREADGROUP BLOCK
# оставляет в соединении consumer'а недочитанный ответ, и следующая команда
# на нём (раньше это был DEL heartbeat-ключа) виснет до таймаута чтения —
# именно этот TimeoutError раньше подменял собой настоящую причину остановки.
r_hb = make_redis()
r_consumer = make_redis()
hb_task = asyncio.create_task(heartbeat_loop(r_hb))
consumer_task = asyncio.create_task(consumer_loop(r_consumer))
try:
    await asyncio.gather(hb_task, consumer_task)
except (KeyboardInterrupt, asyncio.CancelledError):
    print("Останавливаюсь (Interrupt)...")
except Exception:
    print("Воркер упал. Настоящая причина:")
    traceback.print_exc()
finally:
    # Сначала гасим ОБА таска, потом снимаем ключ: иначе при упавшем
    # consumer'е живой heartbeat заново создал бы worker:alive после DEL —
    # зомби-статус воркера, который ничего не потребляет.
    hb_task.cancel()
    consumer_task.cancel()
    await asyncio.gather(hb_task, consumer_task, return_exceptions=True)
    await r_consumer.aclose()   # соединение «грязное» — просто рвём
    try:
        await r_hb.delete(KEY_PREFIX + "worker:alive")
        print("Воркер остановлен, heartbeat снят.")
    except Exception as exc:
        print(f"Воркер остановлен; снять heartbeat не удалось ({exc!r}) — "
              f"ключ истечёт сам не позже чем через {HEARTBEAT_TTL_SEC}с.")
    finally:
        await r_hb.aclose()
